In [3]:
# ============================================================
#  ewl_parallel_nash.py • 4-player EWL sweep with 20–30 core parallelism
#  Now includes tqdm progress bar for runtime monitoring
# ============================================================
import itertools
import math
import numpy as np
import time
import pickle
from concurrent.futures import ProcessPoolExecutor, as_completed
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit_aer import AerSimulator
from tqdm import tqdm  # NEW: progress bar

# ========== CONFIGURATION ==========
NUM_CORES = 30
GRID_THETA = 11
GRID_PHI = 6
k = 4  # player count

# ===== Generate grid =====
theta_vals = np.linspace(0, math.pi, GRID_THETA)
phi_vals = np.linspace(0, math.pi/2, GRID_PHI)
GRID = list(itertools.product(theta_vals, phi_vals))
GRID_LEN = len(GRID)
PROFILE_SPACE = list(itertools.product(range(GRID_LEN), repeat=k))

# ===== Quantum utilities =====
def entangler(n, gamma):
    qc = QuantumCircuit(n)
    for j in range(n):
        for k in range(j + 1, n):
            qc.cx(j, k)
            qc.rz(2 * gamma, k)
            qc.cx(j, k)
    return qc.to_gate()

def payoff_vector(bits):
    n = len(bits)
    d = bits.count("1")
    if d == 0:
        return [3] * n
    if d == n:
        return [1] * n
    return [5 if b == "1" else 0 for b in bits]

# ===== Pre-compile parameterised circuit =====
def build_param_circuit(k):
    theta = [Parameter(f"θ{q}") for q in range(k)]
    phi = [Parameter(f"φ{q}") for q in range(k)]
    qc = QuantumCircuit(k)
    qc.append(entangler(k, math.pi/2), range(k))
    for q in range(k):
        qc.ry(theta[q], q)
        qc.rz(phi[q], q)
    qc.append(entangler(k, -math.pi/2), range(k))
    qc.save_statevector()
    return transpile(qc, AerSimulator(method="statevector")), theta, phi

BASE_CIRC, THETA_PAR, PHI_PAR = build_param_circuit(k)

# ===== Single-task evaluator =====
def evaluate_profile(profile):
    backend = AerSimulator(method="statevector")
    bind = {}
    for q, idx in enumerate(profile):
        θ, φ = GRID[idx]
        bind[THETA_PAR[q]] = θ
        bind[PHI_PAR[q]] = φ
    circ = BASE_CIRC.assign_parameters(bind, inplace=False)
    vec = backend.run(circ).result().get_statevector()
    probs = np.abs(vec)**2
    payoff = np.zeros(k)
    for i, p in enumerate(probs):
        if p < 1e-8:
            continue
        bits = format(i, f"0{k}b")
        payoff += p * np.array(payoff_vector(bits))
    return profile, tuple(payoff.tolist())

# ===== Parallel execution with tqdm =====
def run_parallel():
    t0 = time.perf_counter()
    results = []
    with ProcessPoolExecutor(max_workers=NUM_CORES) as executor:
        futures = {executor.submit(evaluate_profile, prof): prof for prof in PROFILE_SPACE}
        for future in tqdm(as_completed(futures), total=len(PROFILE_SPACE), desc=f"Simulating {len(PROFILE_SPACE):,} profiles..."):
            results.append(future.result())
    print(f"\nFinished in {time.perf_counter() - t0:.2f} seconds")
    return results

if __name__ == "__main__":
    data = run_parallel()
    with open("ewl_k4_parallel_results1.pkl", "wb") as f:
        pickle.dump(data, f)


Simulating 18,974,736 profiles...: 100%|██████████| 18974736/18974736 [42:35<00:00, 7426.36it/s]



Finished in 3036.22 seconds
